In [1]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import matplotlib.pyplot as plt
import joblib
import warnings

# Ignore warnings
warnings.filterwarnings('ignore')

In [2]:
df = pd.read_csv('../data/dataset_train.csv')
df.head()

,session_id,DateTime,user_id,product,campaign_id,webpage_id,product_category_1,product_category_2,user_group_id,gender,age_level,user_depth,city_development_index,var_1,is_click
0,140690,2017-07-02 00:00,858557,C,359520,13787,4,NaN,10.0,Female,4.0,3.0,3.0,0,0
1,333291,2017-07-02 00:00,243253,C,105960,11085,5,NaN,8.0,Female,2.0,2.0,NaN,0,0
2,129781,2017-07-02 00:00,243253,C,359520,13787,4,NaN,8.0,Female,2.0,2.0,NaN,0,0
3,464848,2017-07-02 00:00,1097446,I,359520,13787,3,NaN,3.0,Male,3.0,3.0,2.0,1,0
4,90569,2017-07-02 00:01,663656,C,405490,60305,3,NaN,2.0,Male,2.0,3.0,2.0,1,0


In [ ]:
df['DateTime'] = pd.to_datetime(df['DateTime'])

df["year"] = df["DateTime"].dt.year
df["month"] = df["DateTime"].dt.month
df["day"] = df["DateTime"].dt.day
df["dayofweek"] = df["DateTime"].dt.dayofweek
df["hour"] = df["DateTime"].dt.hour
df["weekofyear"] = df["DateTime"].dt.isocalendar().week
df["is_weekend"] = df["DateTime"].isin([5,6]).astype(int)

df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24)
df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24)

df["dow_sin"] = np.sin(2 * np.pi * df["dayofweek"] / 7)
df["dow_cos"] = np.cos(2 * np.pi * df["dayofweek"] / 7)

df = df.sort_values(["user_id", "DateTime"])

df["lag_1"] = df.groupby("user_id")["is_click"].shift(1)
df["lag_7"] = df.groupby("user_id")["is_click"].shift(7)
df["lag_30"] = df.groupby("user_id")["is_click"].shift(30)

df["rolling_mean_7"] = (
    df.groupby("user_id")["is_click"]
      .shift(1)
      .rolling(7)
      .mean()
)

df["rolling_std_30"] = (
    df.groupby("user_id")["is_click"]
      .shift(1)
      .rolling(30)
      .std()
)

df.head()

In [3]:
from time_feature import TimeFeaturesTransformer

tft = TimeFeaturesTransformer(user_col="user_id", time_col="DateTime", target_col="is_click")

df = tft.transform(df)

df.head()

,session_id,DateTime,user_id,product,campaign_id,webpage_id,product_category_1,product_category_2,user_group_id,gender,...,hour_sin,hour_cos,dow_sin,dow_cos,lag_1,lag_7,lag_30,rolling_mean_7,rolling_std_30,ewm_7
45068,4321,2017-07-02 14:51:00+00:00,4,I,404347,53587,1,146115.0,11.0,Female,...,-5.000000e-01,-0.866025,-0.781831,0.623490,NaN,NaN,NaN,0.000000,0.315302,0.014480
347168,106452,2017-07-06 12:33:00+00:00,6,C,405490,60305,3,NaN,3.0,Male,...,1.224647e-16,-1.000000,0.433884,-0.900969,NaN,NaN,NaN,0.000000,0.323381,0.014655
47703,538078,2017-07-02 15:25:00+00:00,19,E,98970,6970,2,NaN,0.0,Male,...,-7.071068e-01,-0.707107,-0.781831,0.623490,NaN,NaN,NaN,0.000000,0.208514,0.061142
250894,538079,2017-07-05 09:22:00+00:00,19,E,98970,6970,2,NaN,0.0,Male,...,7.071068e-01,-0.707107,0.974928,-0.222521,0.0,NaN,NaN,0.000000,0.374634,0.027269
150872,354959,2017-07-03 20:14:00+00:00,25,B,360936,13787,2,NaN,2.0,Male,...,-8.660254e-01,0.500000,0.000000,1.000000,NaN,NaN,NaN,0.166667,0.200000,0.187512
